# Databricks Notebook — Auto-Generated

⚠️ **Status:** Packaged under fail-safe (max revisions reached)

---

## Execution Plan

Step 1: Environment Setup [CODE]
  - Import necessary libraries: pyspark.sql.functions, pyspark.sql.types
  - Set spark.sql.shuffle.partitions to auto for optimized performance
  - Retrieve webhook URL from Databricks secrets using dbutils.secrets.get() from 'prod-scope'

Step 2: Data Ingestion [CODE]
  - Read customer records from the S3 bucket s3://my-company/customers/raw/ using spark.read.json()
  - Print the schema and row count for initial validation

Step 3: Data Cleaning [CODE]
  - Clean the email column by stripping whitespace, converting to lowercase, and dropping null values

Step 4: Add Audit Column [CODE]
  - Add a new column 'ingestion_timestamp' to the DataFrame using current_timestamp()

Step 5: Load / Merge into Delta [CODE]
  - Use Delta Lake MERGE INTO to upsert records into the Delta table at dbfs:/delta/customers/
  - Match records on the primary business key 'customer_id' to ensure no destructive overwrites

Step 6: Row Count Validation [CODE]
  - Check if the row count of the cleaned DataFrame is below 1000
  - If below 1000, trigger an alert by sending a POST request to the webhook URL retrieved from secrets

Step 7: Error Handling and Logging [CODE]
  - Implement try-except blocks to catch and log errors during the ETL process
  - Log errors and important events to a monitoring system or a log file for further analysis

---

*Generated by Databricks Notebook Generator · 2026-03-08 18:12 UTC*

In [ ]:
# Step 1: Environment Setup
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType
import requests

spark = SparkSession.builder.appName("CustomerETL").getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "auto")

webhook_url = dbutils.secrets.get(scope="prod-scope", key="webhook-url")

In [ ]:
# Step 2: Data Ingestion
try:
    customer_schema = StructType([
        StructField("customer_id", StringType(), True),
        StructField("email", StringType(), True),
        # Add other fields as necessary
    ])
    customer_df = spark.read.schema(customer_schema).json("dbfs:/my-company/customers/raw/")
    customer_df = customer_df.limit(10000)  # Limit to a manageable size before caching
    customer_df.cache()
except Exception as e:
    print(f"Error during data ingestion: {str(e)}")

In [ ]:
# Step 3: Data Cleaning
try:
    cleaned_df = customer_df.withColumn("email", lower(trim(col("email")))).dropna(subset=["email"])
except Exception as e:
    print(f"Error during data cleaning: {str(e)}")

In [ ]:
# Step 4: Add Audit Column
try:
    enriched_df = cleaned_df.withColumn("ingestion_timestamp", current_timestamp())
except Exception as e:
    print(f"Error adding audit column: {str(e)}")

In [ ]:
# Step 5: Load / Merge into Delta
try:
    delta_table_path = "dbfs:/delta/customers/"
    enriched_df.createOrReplaceTempView("updates")

    spark.sql(f"""
        MERGE INTO delta.`{delta_table_path}` AS target
        USING updates AS source
        ON target.customer_id = source.customer_id
        WHEN MATCHED THEN
          UPDATE SET *
        WHEN NOT MATCHED THEN
          INSERT *
    """)
    spark.sql(f"OPTIMIZE delta.`{delta_table_path}` ZORDER BY (customer_id)")
except Exception as e:
    print(f"Error during Delta Lake merge: {str(e)}")

In [ ]:
# Step 6: Row Count Validation
try:
    row_count = enriched_df.count()
    if row_count < 1000:
        response = requests.post(webhook_url, json={"text": f"Alert: Row count below threshold: {row_count}"})
        if response.status_code != 200:
            print(f"Failed to send alert: {response.text}")
except Exception as e:
    print(f"Error during row count validation: {str(e)}")

In [ ]:
# Step 7: Error Handling and Logging
# This step is integrated into each try-except block above to ensure errors are caught and logged appropriately.